In [1]:
# ============================================================
# FIXMATCH ON MNIST
# 90/10 TO 10/90 LABELED / UNLABELED SPLITS
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms

import numpy as np
import pandas as pd
import random
import gc


# ============================================================
# 1. SETTINGS
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

BATCH_SIZE = 128

EPOCHS_INITIAL = 3
EPOCHS_FIXMATCH = 3

THRESHOLD = 0.95

LAMBDA_U = 1.0


# ============================================================
# 2. TRANSFORMS
# ============================================================

# Weak augmentation
weak_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor()
])


# Strong augmentation
strong_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ToTensor()
])


# Basic transform for storing original images
base_transform = transforms.ToTensor()


# ============================================================
# 3. MNIST DATASET
# ============================================================

train_dataset = datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=base_transform
)

test_dataset = datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=base_transform
)

print(
    "Total training samples:",
    len(train_dataset)
)

print(
    "Total test samples:",
    len(test_dataset)
)


# ============================================================
# 4. CUSTOM DATASET FOR FIXMATCH
# ============================================================

class FixMatchLabeledDataset(Dataset):

    def __init__(
        self,
        dataset,
        indices
    ):

        self.dataset = dataset
        self.indices = indices

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, index):

        real_index = self.indices[index]

        image, label = self.dataset[
            real_index
        ]

        # Weak version for supervised training
        weak_image = weak_transform(
            transforms.ToPILImage()(image)
        )

        return weak_image, label


class FixMatchUnlabeledDataset(Dataset):

    def __init__(
        self,
        dataset,
        indices
    ):

        self.dataset = dataset
        self.indices = indices

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, index):

        real_index = self.indices[index]

        image, _ = self.dataset[
            real_index
        ]

        # Return ORIGINAL image.
        # Weak and strong augmentations are generated
        # separately inside the training loop.

        return image


# ============================================================
# 5. CNN MODEL
# ============================================================

class CNN(nn.Module):

    def __init__(self):

        super(CNN, self).__init__()

        self.conv = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 7 * 7,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                10
            )
        )

    def forward(self, x):

        x = self.conv(x)

        x = self.fc(x)

        return x


# ============================================================
# 6. TEST ACCURACY
# ============================================================

test_loader = DataLoader(

    test_dataset,

    batch_size=BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=(
        device.type == "cuda"
    )
)


def test_accuracy(model):

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return (
        100.0 * correct / total
    )


# ============================================================
# 7. INITIAL SUPERVISED TRAINING
# ============================================================

def initial_training(
    model,
    labeled_loader,
    epochs
):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0

        for images, labels in labeled_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            loss.backward()

            optimizer.step()

            total_loss += (
                loss.item()
                *
                images.size(0)
            )

        avg_loss = (
            total_loss /
            len(labeled_loader.dataset)
        )

        print(
            f"      Epoch "
            f"{epoch + 1}/{epochs} "
            f"Loss: {avg_loss:.4f}"
        )

    return model


# ============================================================
# 8. FIXMATCH TRAINING
# ============================================================

def fixmatch_train(
    model,
    labeled_loader,
    unlabeled_loader,
    epochs,
    threshold=0.95,
    lambda_u=1.0
):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(
        model.parameters(),
        lr=0.001
    )

    model.train()

    for epoch in range(epochs):

        total_loss = 0.0
        total_supervised = 0.0
        total_unsupervised = 0.0

        total_unlabeled = 0
        total_selected = 0

        unlabeled_iter = iter(
            unlabeled_loader
        )

        for images_l, labels_l in labeled_loader:

            # =================================================
            # LABELED DATA
            # =================================================

            images_l = images_l.to(
                device,
                non_blocking=True
            )

            labels_l = labels_l.to(
                device,
                non_blocking=True
            )

            # =================================================
            # SUPERVISED LOSS
            # =================================================

            outputs_l = model(
                images_l
            )

            loss_sup = criterion(
                outputs_l,
                labels_l
            )

            # =================================================
            # GET UNLABELED DATA
            # =================================================

            try:

                images_u = next(
                    unlabeled_iter
                )

            except StopIteration:

                unlabeled_iter = iter(
                    unlabeled_loader
                )

                images_u = next(
                    unlabeled_iter
                )

            images_u = images_u.to(
                device,
                non_blocking=True
            )

            total_unlabeled += (
                images_u.size(0)
            )

            # =================================================
            # CREATE WEAK AUGMENTATION
            # =================================================

            weak_images = torch.stack([

                weak_transform(
                    transforms.ToPILImage()(img.cpu())
                )

                for img in images_u

            ]).to(device)

            # =================================================
            # CREATE STRONG AUGMENTATION
            # =================================================

            strong_images = torch.stack([

                strong_transform(
                    transforms.ToPILImage()(img.cpu())
                )

                for img in images_u

            ]).to(device)

            # =================================================
            # GENERATE PSEUDO-LABELS
            # =================================================

            with torch.no_grad():

                weak_outputs = model(
                    weak_images
                )

                weak_probs = torch.softmax(
                    weak_outputs,
                    dim=1
                )

                confidence, pseudo_labels = torch.max(
                    weak_probs,
                    dim=1
                )

                mask = (
                    confidence >= threshold
                )

            # =================================================
            # UNSUPERVISED LOSS
            # =================================================

            if mask.any():

                strong_outputs = model(
                    strong_images
                )

                loss_unsup = criterion(

                    strong_outputs[mask],

                    pseudo_labels[mask]

                )

                total_selected += (
                    mask.sum().item()
                )

                loss = (
                    loss_sup
                    +
                    lambda_u *
                    loss_unsup
                )

            else:

                loss_unsup = torch.tensor(
                    0.0,
                    device=device
                )

                loss = loss_sup

            # =================================================
            # BACKPROPAGATION
            # =================================================

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            # =================================================
            # LOSS TRACKING
            # =================================================

            total_loss += (
                loss.item()
            )

            total_supervised += (
                loss_sup.item()
            )

            total_unsupervised += (
                loss_unsup.item()
            )

        num_batches = len(
            labeled_loader
        )

        selection_rate = (
            100.0 *
            total_selected /
            max(total_unlabeled, 1)
        )

        print(

            f"      Epoch "
            f"{epoch + 1}/{epochs} | "

            f"Total Loss: "
            f"{total_loss / num_batches:.4f} | "

            f"Supervised: "
            f"{total_supervised / num_batches:.4f} | "

            f"Unsupervised: "
            f"{total_unsupervised / num_batches:.4f} | "

            f"Pseudo-label selection: "
            f"{selection_rate:.2f}%"

        )

    return model


# ============================================================
# 9. EXPERIMENTAL SPLITS
# ============================================================

split_fractions = [

    0.90,
    0.80,
    0.70,
    0.60,
    0.50,
    0.40,
    0.30,
    0.20,
    0.10

]


# ============================================================
# 10. RESULTS
# ============================================================

all_results = []

total_samples = len(
    train_dataset
)


# ============================================================
# 11. RUN ALL NINE SPLITS
# ============================================================

for labeled_fraction in split_fractions:

    # ========================================================
    # SPLIT NAME
    # ========================================================

    labeled_percent = int(
        labeled_fraction * 100
    )

    unlabeled_percent = (
        100 -
        labeled_percent
    )

    split_name = (
        f"{labeled_percent}/"
        f"{unlabeled_percent}"
    )

    print("\n" + "=" * 70)

    print(
        f"EXPERIMENT: "
        f"{split_name} "
        f"(LABELED / UNLABELED)"
    )

    print("=" * 70)


    # ========================================================
    # CREATE SAME TYPE OF RANDOM SPLIT
    # ========================================================

    rng = np.random.default_rng(
        SEED
    )

    indices = np.arange(
        total_samples
    )

    rng.shuffle(indices)

    num_labeled = int(
        total_samples *
        labeled_fraction
    )

    labeled_idx = (
        indices[
            :num_labeled
        ]
    )

    unlabeled_idx = (
        indices[
            num_labeled:
        ]
    )

    print(
        f"Initial labeled samples: "
        f"{len(labeled_idx)}"
    )

    print(
        f"Initial unlabeled samples: "
        f"{len(unlabeled_idx)}"
    )


    # ========================================================
    # DATASETS
    # ========================================================

    labeled_dataset = (
        FixMatchLabeledDataset(
            train_dataset,
            labeled_idx
        )
    )

    unlabeled_dataset = (
        FixMatchUnlabeledDataset(
            train_dataset,
            unlabeled_idx
        )
    )


    # ========================================================
    # DATALOADERS
    # ========================================================

    labeled_loader = DataLoader(

        labeled_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )

    unlabeled_loader = DataLoader(

        unlabeled_dataset,

        batch_size=BATCH_SIZE,

        shuffle=True,

        num_workers=0,

        pin_memory=(
            device.type == "cuda"
        )
    )


    # ========================================================
    # NEW MODEL FOR EACH SPLIT
    # ========================================================

    model = CNN().to(device)


    # ========================================================
    # INITIAL SUPERVISED TRAINING
    # ========================================================

    print(
        "\nInitial supervised training..."
    )

    model = initial_training(

        model,

        labeled_loader,

        EPOCHS_INITIAL

    )


    # ========================================================
    # INITIAL ACCURACY
    # ========================================================

    initial_accuracy = test_accuracy(
        model
    )

    print(
        f"Initial Test Accuracy: "
        f"{initial_accuracy:.2f}%"
    )


    all_results.append({

        "split":
            split_name,

        "stage":
            "Initial",

        "accuracy_percent":
            round(
                initial_accuracy,
                2
            )

    })


    # ========================================================
    # FIXMATCH
    # ========================================================

    print(
        "\n--- FixMatch Training ---"
    )

    print(
        f"Confidence threshold = "
        f"{THRESHOLD}"
    )

    print(
        f"Unsupervised weight = "
        f"{LAMBDA_U}"
    )


    model = fixmatch_train(

        model,

        labeled_loader,

        unlabeled_loader,

        EPOCHS_FIXMATCH,

        threshold=THRESHOLD,

        lambda_u=LAMBDA_U

    )


    # ========================================================
    # FINAL ACCURACY
    # ========================================================

    final_accuracy = test_accuracy(
        model
    )

    print(
        f"FixMatch Test Accuracy: "
        f"{final_accuracy:.2f}%"
    )


    all_results.append({

        "split":
            split_name,

        "stage":
            "FixMatch",

        "accuracy_percent":
            round(
                final_accuracy,
                2
            )

    })


    # ========================================================
    # MEMORY CLEANUP
    # ========================================================

    del model

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


# ============================================================
# 12. FINAL RESULTS
# ============================================================

fixmatch_results = pd.DataFrame(
    all_results
)

print("\n" + "=" * 70)

print(
    "FINAL FIXMATCH RESULTS"
)

print("=" * 70)

display(
    fixmatch_results
)


# ============================================================
# 13. FIXMATCH ONLY
# ============================================================

final_fixmatch = (
    fixmatch_results[
        fixmatch_results["stage"]
        ==
        "FixMatch"
    ]
    .copy()
)

print(
    "\nFinal FixMatch accuracy "
    "for each split:"
)

display(
    final_fixmatch
)


# ============================================================
# 14. SAVE RESULTS
# ============================================================

fixmatch_results.to_csv(
    "fixmatch_results.csv",
    index=False
)

final_fixmatch.to_csv(
    "fixmatch_final_results.csv",
    index=False
)

print(
    "\nResults saved:"
)

print(
    "fixmatch_results.csv"
)

print(
    "fixmatch_final_results.csv"
)

Device: cuda


100%|██████████| 9.91M/9.91M [00:00<00:00, 56.3MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.70MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 14.9MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.32MB/s]


Total training samples: 60000
Total test samples: 10000

EXPERIMENT: 90/10 (LABELED / UNLABELED)
Initial labeled samples: 54000
Initial unlabeled samples: 6000

Initial supervised training...
      Epoch 1/3 Loss: 0.4229
      Epoch 2/3 Loss: 0.1318
      Epoch 3/3 Loss: 0.0882
Initial Test Accuracy: 97.65%

--- FixMatch Training ---
Confidence threshold = 0.95
Unsupervised weight = 1.0
      Epoch 1/3 | Total Loss: 0.1017 | Supervised: 0.0711 | Unsupervised: 0.0306 | Pseudo-label selection: 91.94%
      Epoch 2/3 | Total Loss: 0.0757 | Supervised: 0.0547 | Unsupervised: 0.0210 | Pseudo-label selection: 94.04%
      Epoch 3/3 | Total Loss: 0.0639 | Supervised: 0.0452 | Unsupervised: 0.0187 | Pseudo-label selection: 95.22%
FixMatch Test Accuracy: 98.51%

EXPERIMENT: 80/20 (LABELED / UNLABELED)
Initial labeled samples: 48000
Initial unlabeled samples: 12000

Initial supervised training...
      Epoch 1/3 Loss: 0.4432
      Epoch 2/3 Loss: 0.1384
      Epoch 3/3 Loss: 0.0868
Initial Test 

,split,stage,accuracy_percent
0,90/10,Initial,97.65
1,90/10,FixMatch,98.51
2,80/20,Initial,97.79
3,80/20,FixMatch,98.23
4,70/30,Initial,97.26
5,70/30,FixMatch,98.10
6,60/40,Initial,96.55
7,60/40,FixMatch,98.06
8,50/50,Initial,96.84
9,50/50,FixMatch,97.96



Final FixMatch accuracy for each split:


,split,stage,accuracy_percent
1,90/10,FixMatch,98.51
3,80/20,FixMatch,98.23
5,70/30,FixMatch,98.10
7,60/40,FixMatch,98.06
9,50/50,FixMatch,97.96
11,40/60,FixMatch,97.14
13,30/70,FixMatch,96.39
15,20/80,FixMatch,96.01
17,10/90,FixMatch,93.30



Results saved:
fixmatch_results.csv
fixmatch_final_results.csv
